In [2]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_friedman1
import matplotlib.pyplot as plt

import sys

sys.path.append("../..")

In [3]:
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.influence_func_cp import InfluenceFunctionConformalPredictor
from src.conformal_prediction.utils import interval_length

## Example of run
### Sample dataset

In [4]:
sample_size = 500
sample_size_p1 = sample_size + 1

input_points, output_points = make_friedman1(sample_size_p1)
output_points = output_points.reshape(-1, 1)

input_points = StandardScaler().fit_transform(input_points)
output_points = StandardScaler().fit_transform(output_points)

train_input_points = input_points[:-1, :].reshape(sample_size, input_points.shape[1])
train_output_points = output_points[:-1, :].reshape(sample_size, output_points.shape[1])
test_input_point = input_points[-1, :].reshape(1, -1)
test_output_point = output_points[-1, :].reshape(1, -1)

### Instantiate prediction region

In [5]:
loss_name = "log_cosh"
loss_params = {"alpha": 1.0}

# loss_name = "pseudo_huber"
# loss_params = {"alpha": 1.}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    lam=(1 / sample_size) ** (0.33),
    kernel="laplacian",
    loss_name=loss_name,
    loss_params=loss_params,
)

conformal_predictor = conformal_predictor = InfluenceFunctionConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_point
)

### Compute prediction region and bounds on thickness

In [7]:
confidence_control_level = 0.1
prediction_region = region_predictor(confidence_control_level)

In [9]:
prediction_region

[{'upper': [-1.6795555635378556,1.6136221995226374],
  'lower': [-1.6779286958459332,1.6121489105973454]}]

In [11]:
empirical_thickness_bound = interval_length(
    prediction_region[0]["upper"] - prediction_region[0]["lower"]
)

In [12]:
print(empirical_thickness_bound)

0.0031001566172144113
